# SIC 2026 · Inteligencia Artificial
## Clase 17: Redes generativas: autoencoders y GAN (Capítulo 9, Unidad 3)

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Hasta ahora todas las redes **clasificaban o predecían**. Hoy vemos redes que **reconstruyen y generan** imágenes.

| Hora | Parte | Láminas | Qué vamos a hacer |
|---|---|---|---|
| 18:00 a 18:45 | 1. Autoencoder | 82 a 89 | Comprimir prendas a 32 números y reconstruirlas |
| 18:45 a 19:30 | 2. Usos del autoencoder | | Quitar ruido de imágenes y **detectar anomalías** |
| 19:30 a 19:40 | Pausa | | |
| 19:40 a 20:20 | 3. GAN: la idea | 91 a 104 | Falsificador contra detective, con la pérdida paso a paso |
| 20:20 a 21:50 | 4. GAN: el ejercicio | 105 a 132 | Entrenar una GAN que genera prendas de ropa |
| 21:50 a 22:00 | Cierre | | |

**Antes de empezar:** *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)*. La Parte 4 lo necesita.

Hay 10 ejercicios con verificación; las soluciones están al final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
import time, warnings
warnings.filterwarnings("ignore")

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

def mostrar(filas, titulos=None, n=8, alto=1.6):
    """Muestra varias filas de imágenes de 28 x 28, una fila por lista."""
    fig, ejes = plt.subplots(len(filas), n, figsize=(n * 1.3, alto * len(filas)))
    ejes = np.atleast_2d(ejes)
    for f, imagenes in enumerate(filas):
        for k in range(n):
            ejes[f, k].imshow(np.asarray(imagenes[k]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
            ejes[f, k].axis("off")
        if titulos: ejes[f, 0].set_title(titulos[f], fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = (X_tr / 255.0).astype("float32").reshape(-1, 784)
X_te = (X_te / 255.0).astype("float32").reshape(-1, 784)
PRENDAS = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo", "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]
print("Entrenamiento:", X_tr.shape, "| Prueba:", X_te.shape, "| GPU:", tf.config.list_physical_devices("GPU") or "no")

---
# Parte 1. Autoencoder (láminas 82 a 89)

> **Autoencoder (codificador automático):** una red que aprende a **copiar su entrada en su salida**, pero obligada a pasar por un **cuello de botella** de pocos números.

Parece una tarea tonta: ¿para qué copiar una imagen? El truco está en el cuello de botella. Para reconstruir 784 pixeles a partir de solo 32 números, la red tiene que aprender **lo esencial** de las imágenes: qué forma tiene una camiseta, qué distingue a un zapato.

| Parte | Qué hace | En nuestra red |
|---|---|---|
| **Codificador** (encoder) | Comprime la imagen en pocos números | 784 → 128 → 32 |
| **Código** (espacio latente) | La versión comprimida: 32 números que resumen la prenda | 32 |
| **Decodificador** (decoder) | Reconstruye la imagen desde el código | 32 → 128 → 784 |

**No necesita etiquetas:** la respuesta correcta es la misma imagen. Es aprendizaje **no supervisado**, como el clustering del Capítulo 6.

## Paso 1.1. Construir el autoencoder

Lo armamos como dos modelos (codificador y decodificador) unidos, para poder usar cada mitad por separado después.

In [ ]:
CODIGO = 32

keras.utils.set_random_seed(42)
codificador = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(CODIGO, activation="relu"),           # el cuello de botella
], name="codificador")

decodificador = keras.Sequential([
    layers.Input(shape=(CODIGO,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(784, activation="sigmoid"),           # sigmoide: pixeles entre 0 y 1
], name="decodificador")

autoencoder = keras.Sequential([codificador, decodificador], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()

**Qué deberías ver:** dos bloques, el codificador y el decodificador, con cerca de 210.000 parámetros en total (casi todos en las dos capas que conectan con los 784 pixeles).

**¿Por qué sigmoide y `binary_crossentropy`?** Cada pixel está entre 0 y 1, como una probabilidad. La sigmoide asegura salidas en ese rango, y la entropía cruzada compara pixel por pixel. También funciona con `"mse"`.

### Ejercicio 1
¿Cuántas veces más pequeño es el código que la imagen? Guarda en `factor_compresion` la división entre los pixeles de la imagen y los números del código.

In [ ]:
factor_compresion = None  # TU CÓDIGO AQUÍ
print(factor_compresion)

In [ ]:
revisar(lambda: np.isclose(factor_compresion, 784 / 32))

## Paso 1.2. Entrenar

Fíjense en el `fit`: **la entrada y la respuesta son lo mismo**, `X_tr` y `X_tr`.

In [ ]:
historia = autoencoder.fit(X_tr, X_tr, epochs=10, batch_size=256, validation_data=(X_te, X_te), verbose=0)
print("Pérdida final en validación:", round(historia.history["val_loss"][-1], 4))

reconstruidas = autoencoder.predict(X_te[:8], verbose=0)
mostrar([X_te[:8], reconstruidas], ["Originales", "Reconstruidas desde 32 números"])

**Qué deberías ver:** reconstrucciones reconocibles pero **borrosas**: se mantiene la forma de cada prenda y se pierden detalles finos (estampados, texturas). Es el precio de comprimir 24 veces.

### Ejercicio 2
Usa solo el **codificador** para obtener los códigos de las 8 primeras imágenes de prueba, y guárdalos en `codigos_8`. ¿Qué forma tienen?

In [ ]:
codigos_8 = None  # TU CÓDIGO AQUÍ
print(codigos_8.shape if codigos_8 is not None else None)

In [ ]:
revisar(lambda: codigos_8.shape == (8, CODIGO))

## Paso 1.3. ¿El código guarda información útil?

Si el código resume lo esencial, prendas del mismo tipo deberían tener códigos parecidos. Lo comprobamos de dos formas: **dibujando** los códigos en 2D (con PCA, Clase 8) y **clasificando** con ellos.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

codigos_tr = codificador.predict(X_tr, verbose=0)
codigos_te = codificador.predict(X_te, verbose=0)

Z = PCA(n_components=2).fit_transform(codigos_te[:3000])
plt.figure(figsize=(8, 5.5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=y_te[:3000], cmap="tab10", s=6)
plt.legend(sc.legend_elements()[0], PRENDAS, fontsize=7, loc="best"); plt.title("Códigos de 32 números, vistos en 2D")
plt.show()

exac_codigo = accuracy_score(y_te, LogisticRegression(max_iter=2000).fit(codigos_tr, y_tr).predict(codigos_te))
print(f"Logística con los 32 números del código: {exac_codigo:.3f}")

**Qué deberías ver:** grupos de colores separados: zapatos por un lado, pantalones por otro, y las prendas superiores mezcladas entre sí. Una logística con solo los 32 números del código llega a cerca de **0,8**, aunque el autoencoder **nunca vio las etiquetas**.

Esa es la utilidad de fondo: **el codificador aprendió una representación compacta y con sentido**, como los embeddings de palabras de la Clase 13, pero para imágenes.

---
# Parte 2. Usos del autoencoder

## Paso 2.1. Quitar ruido (denoising autoencoder)

**La idea:** entrenar con imágenes **con ruido** como entrada, y las **limpias** como respuesta. La red aprende a separar la prenda del ruido.

In [ ]:
rng = np.random.default_rng(0)
ruido = 0.4
X_tr_ruido = np.clip(X_tr + ruido * rng.normal(size=X_tr.shape), 0, 1).astype("float32")
X_te_ruido = np.clip(X_te + ruido * rng.normal(size=X_te.shape), 0, 1).astype("float32")

keras.utils.set_random_seed(42)
limpiador = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation="relu"), layers.Dense(64, activation="relu"),
    layers.Dense(256, activation="relu"), layers.Dense(784, activation="sigmoid"),
])
limpiador.compile(optimizer="adam", loss="binary_crossentropy")
limpiador.fit(X_tr_ruido, X_tr, epochs=10, batch_size=256, verbose=0)     # entrada con ruido, respuesta limpia

mostrar([X_te_ruido[:8], limpiador.predict(X_te_ruido[:8], verbose=0), X_te[:8]],
        ["Con ruido (entrada)", "Limpiadas por la red", "Originales"])

**Qué deberías ver:** imágenes muy ruidosas que la red devuelve limpias y reconocibles. Se usa para restaurar fotos antiguas, escaneos o imágenes médicas.

### Ejercicio 3
Calcula el error cuadrático medio (MSE) entre las imágenes **con ruido** y las originales, y entre las **limpiadas** y las originales, sobre las 1.000 primeras de prueba. Guárdalos en `mse_ruido` y `mse_limpia`. ¿Cuánto bajó el error?

In [ ]:
mse_ruido = None   # TU CÓDIGO AQUÍ
mse_limpia = None  # TU CÓDIGO AQUÍ
print(mse_ruido, mse_limpia)

In [ ]:
revisar(lambda: np.isclose(mse_ruido, np.mean((X_te_ruido[:1000] - X_te[:1000]) ** 2), rtol=1e-4) and mse_limpia < mse_ruido)

## Paso 2.2. Detectar anomalías

Este es uno de los usos más aplicados de los autoencoders, y sirve para proyectos reales: **fraudes, fallas de máquinas, defectos de fabricación**.

**La idea:**

1. Entrenar el autoencoder **solo con casos normales**.
2. Aprende a reconstruir bien **lo normal**.
3. Cuando llega algo distinto, lo reconstruye **mal**: su **error de reconstrucción** es alto.
4. Ese error es una **señal de anomalía**.

**Nuestro caso:** una fábrica de calzado deportivo. Lo "normal" son las **zapatillas**. Queremos detectar cualquier otra cosa que aparezca en la línea de producción.

In [ ]:
NORMAL = 7                                           # Zapatilla
X_normal = X_tr[y_tr == NORMAL]
print("Zapatillas para entrenar:", len(X_normal))

keras.utils.set_random_seed(42)
detector = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(64, activation="relu"), layers.Dense(16, activation="relu"),
    layers.Dense(64, activation="relu"), layers.Dense(784, activation="sigmoid"),
])
detector.compile(optimizer="adam", loss="mse")
detector.fit(X_normal, X_normal, epochs=20, batch_size=128, verbose=0)

def error_reconstruccion(X):
    return np.mean((X - detector.predict(X, verbose=0)) ** 2, axis=1)     # un error por imagen

errores = error_reconstruccion(X_te)
es_anomalia = (y_te != NORMAL).astype(int)            # 1 si NO es zapatilla

plt.figure(figsize=(8, 4))
plt.hist(errores[es_anomalia == 0], bins=60, alpha=0.6, label="Zapatillas (normales)", density=True)
plt.hist(errores[es_anomalia == 1], bins=60, alpha=0.6, label="Otras prendas (anomalías)", density=True)
plt.xlabel("Error de reconstrucción"); plt.legend(); plt.title("Las anomalías se reconstruyen peor"); plt.show()

**Qué deberías ver:** las zapatillas con errores bajos, concentradas a la izquierda, y las otras prendas con errores más altos, a la derecha.

**¿Dónde poner el umbral?** Una regla común: el percentil 95 de los errores de los casos **normales**. Así, solo un 5% de las zapatillas buenas dispararía una falsa alarma.

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score

errores_normales_tr = error_reconstruccion(X_normal)
umbral = np.percentile(errores_normales_tr, 95)
alarma = (errores > umbral).astype(int)

print(f"Umbral (percentil 95 de las zapatillas de entrenamiento): {umbral:.4f}")
print(f"ROC AUC del error como detector: {roc_auc_score(es_anomalia, errores):.3f}")
print(f"Recall (anomalías detectadas): {recall_score(es_anomalia, alarma):.3f}")
print(f"Falsas alarmas en zapatillas: {alarma[es_anomalia == 0].mean():.3f}")

peores = np.argsort(errores)[::-1][:8]
mostrar([X_te[peores], detector.predict(X_te[peores], verbose=0)], ["Mayor error: lo más 'raro'", "Cómo las reconstruye"])
print("Qué son:", [PRENDAS[y_te[i]] for i in peores])

**Qué deberías ver:** un ROC AUC alto (sobre 0,9), la mayoría de las anomalías detectadas y cerca de un 5% de falsas alarmas, como fijamos con el percentil 95. En las imágenes con mayor error, el detector **intenta dibujar una zapatilla** sobre algo que no lo es: por eso el error es tan alto.

**Lo valioso:** el detector **nunca vio una anomalía** al entrenar. En fraude o fallas eso es clave, porque los casos anormales son pocos y cambian todo el tiempo.

### Ejercicio 4
¿Qué prenda es la **más difícil** de distinguir de una zapatilla? Calcula el **error de reconstrucción promedio** de cada prenda en `X_te` y guarda en `prenda_mas_parecida` el nombre de la prenda (distinta de Zapatilla) con **menor** error promedio.

In [ ]:
prenda_mas_parecida = None  # TU CÓDIGO AQUÍ
print(prenda_mas_parecida)

In [ ]:
_p = pd.Series(errores).groupby(y_te).mean().drop(NORMAL)
revisar(lambda: prenda_mas_parecida == PRENDAS[_p.idxmin()])

**Qué deberías ver:** casi siempre **Sandalia** o **Botín**: también son calzado. Tiene sentido.

### Ejercicio 5
Si la fábrica prefiere **no dejar pasar casi ninguna anomalía**, aunque haya más falsas alarmas, ¿el umbral debe subir o bajar? Prueba con el percentil **80** de las zapatillas de entrenamiento y guarda el nuevo recall en `recall_p80`.

In [ ]:
recall_p80 = None  # TU CÓDIGO AQUÍ
print(recall_p80)

In [ ]:
revisar(lambda: np.isclose(recall_p80, recall_score(es_anomalia, (errores > np.percentile(errores_normales_tr, 80)).astype(int))))

---
# Parte 3. GAN: la idea (láminas 91 a 104)

El autoencoder **reconstruye** imágenes que ya existen. Una **GAN** (red generativa adversaria) **crea imágenes nuevas** que nunca existieron.

## Paso 3.1. Falsificador contra detective

Una GAN tiene **dos redes que compiten**:

| Red | Rol | Recibe | Entrega | Quiere |
|---|---|---|---|---|
| **Generador** | El falsificador | Ruido al azar (un vector de 100 números) | Una imagen falsa | Engañar al discriminador |
| **Discriminador** | El detective | Una imagen, real o falsa | La probabilidad de que sea real | No dejarse engañar |

Entrenan **por turnos**, y cada uno mejora porque el otro mejora:

1. Al principio el generador dibuja manchas, y el discriminador aprende fácil a reconocerlas.
2. El generador ajusta sus pesos para que sus manchas **parezcan más reales** al discriminador.
3. El discriminador tiene que afinar su ojo para seguir distinguiendo.
4. Así, ronda a ronda, las imágenes falsas se vuelven cada vez más realistas.

**El ruido de entrada** (lámina 97) es la "semilla" de cada imagen: ruidos distintos producen prendas distintas.

## Paso 3.2. Las pérdidas (láminas 99 a 102)

Las dos redes usan entropía cruzada binaria (Clase 15), con objetivos opuestos:

**Discriminador:** quiere decir 1 a las reales y 0 a las falsas.

$$L_D = \text{BCE}(1, D(\text{real})) + \text{BCE}(0, D(G(z)))$$

**Generador:** quiere que el discriminador diga **1** a sus imágenes falsas.

$$L_G = \text{BCE}(1, D(G(z)))$$

donde $z$ es el ruido, $G(z)$ la imagen falsa y $D(\cdot)$ la probabilidad de "real" que entrega el discriminador.

El generador **nunca ve imágenes reales**: aprende solo a través de lo que el discriminador opina de sus falsificaciones.

### Ejercicio 6
Para fijar la idea de la pérdida, calcula a mano con `np.log` la entropía cruzada binaria $-[y \log(p) + (1-y)\log(1-p)]$ en dos casos:

- El discriminador recibe una imagen **falsa** ($y = 0$) y dice $p = 0{,}9$ (cree que es real). Guarda la pérdida en `bce_engañado`.
- Recibe una falsa y dice $p = 0{,}1$ (la descubre). Guarda la pérdida en `bce_acierta`.

In [ ]:
bce_engañado = None  # TU CÓDIGO AQUÍ
bce_acierta = None   # TU CÓDIGO AQUÍ
print(bce_engañado, bce_acierta)

In [ ]:
revisar(lambda: np.isclose(bce_engañado, -np.log(0.1)) and np.isclose(bce_acierta, -np.log(0.9)))

**Qué deberías ver:** 2,30 cuando se deja engañar y 0,11 cuando acierta. La pérdida castiga mucho los errores con confianza.

---
# Parte 4. GAN: el ejercicio (láminas 105 a 132)

Seguimos la estructura de las láminas, con Keras: definir el generador y el discriminador, las pérdidas, el paso de entrenamiento con `GradientTape` (Clase 14) y el ciclo.

## Paso 4.1. Los datos

Para la GAN, las imágenes van de **−1 a 1** en vez de 0 a 1: así calzan con la salida `tanh` del generador.

In [ ]:
RUIDO = 100
LOTE = 128
X_gan = (X_tr.reshape(-1, 28, 28, 1) * 2.0 - 1.0).astype("float32")     # de [0, 1] a [-1, 1]
datos_gan = tf.data.Dataset.from_tensor_slices(X_gan).shuffle(60_000, seed=0).batch(LOTE, drop_remainder=True)
print("Lotes por época:", len(list(datos_gan)))

## Paso 4.2. El generador (lámina 111)

De un vector de ruido a una imagen de 28 × 28. Usamos capas **`Conv2DTranspose`**, que hacen lo contrario de una convolución con paso 2: **duplican** el tamaño de la imagen (7 → 14 → 28).

In [ ]:
keras.utils.set_random_seed(42)
generador = keras.Sequential([
    layers.Input(shape=(RUIDO,)),
    layers.Dense(7 * 7 * 64), layers.BatchNormalization(), layers.LeakyReLU(0.2),
    layers.Reshape((7, 7, 64)),                                                   # de vector a "imagen" pequeña
    layers.Conv2DTranspose(64, (4, 4), strides=2, padding="same"), layers.BatchNormalization(), layers.LeakyReLU(0.2),  # 14 x 14
    layers.Conv2DTranspose(1, (4, 4), strides=2, padding="same", activation="tanh"),                                   # 28 x 28
], name="generador")
generador.summary()

prueba = generador(tf.random.normal((8, RUIDO)), training=False)
mostrar([(prueba.numpy() + 1) / 2], ["El generador sin entrenar: puro ruido"])

**Qué deberías ver:** imágenes de 28 × 28 que son solo ruido gris. Todavía no aprendió nada.

## Paso 4.3. El discriminador (lámina 111)

Una CNN de clasificación binaria, como las de la Clase 15: recibe una imagen y entrega un número. Sin sigmoide al final: entrega un **logit**, y la pérdida aplica la sigmoide por dentro (`from_logits=True`), que es más estable numéricamente.

In [ ]:
discriminador = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(64, (4, 4), strides=2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),    # 14 x 14
    layers.Conv2D(128, (4, 4), strides=2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),   # 7 x 7
    layers.Flatten(),
    layers.Dense(1),                                                                                      # logit: real o falsa
], name="discriminador")
discriminador.summary()

### Ejercicio 7
¿Cuántos parámetros tiene la primera `Conv2D(64, (4, 4))` del discriminador, que recibe imágenes de 1 canal? Calcúlalo con la fórmula de la Clase 15 y guárdalo en `params_conv_d`.

In [ ]:
params_conv_d = None  # TU CÓDIGO AQUÍ
print(params_conv_d)

In [ ]:
revisar(lambda: params_conv_d == 64 * (4 * 4 * 1 + 1))

## Paso 4.4. Pérdidas y optimizadores (lámina 124)

Las fórmulas del Paso 3.2, en código. Cada red tiene **su propio optimizador**, porque se entrenan por separado.

In [ ]:
bce = keras.losses.BinaryCrossentropy(from_logits=True)

def perdida_discriminador(logits_reales, logits_falsos):
    return bce(tf.ones_like(logits_reales), logits_reales) + bce(tf.zeros_like(logits_falsos), logits_falsos)

def perdida_generador(logits_falsos):
    return bce(tf.ones_like(logits_falsos), logits_falsos)      # quiere que digan "real" (1)

opt_g = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
opt_d = keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

`tf.ones_like(x)` crea un tensor de unos con la misma forma que `x`: las etiquetas "real". `tf.zeros_like`, las etiquetas "falsa". Los valores `learning_rate=2e-4` y `beta_1=0.5` son los recomendados para GAN: las dos redes deben aprender a un ritmo parecido para que ninguna aplaste a la otra.

## Paso 4.5. Un paso de entrenamiento (láminas 125 y 126)

Es el ciclo de la Clase 14 con `GradientTape`, pero con **dos cintas**: una para cada red.

In [ ]:
@tf.function                                                  # compila la función para que sea mucho más rápida
def paso(imagenes_reales):
    ruido = tf.random.normal((LOTE, RUIDO))
    with tf.GradientTape() as cinta_g, tf.GradientTape() as cinta_d:
        falsas = generador(ruido, training=True)                         # 1. el generador falsifica
        logits_reales = discriminador(imagenes_reales, training=True)    # 2. el discriminador opina de las reales
        logits_falsos = discriminador(falsas, training=True)             #    y de las falsas
        loss_g = perdida_generador(logits_falsos)                        # 3. las dos pérdidas
        loss_d = perdida_discriminador(logits_reales, logits_falsos)
    grad_g = cinta_g.gradient(loss_g, generador.trainable_variables)     # 4. gradientes de cada red
    grad_d = cinta_d.gradient(loss_d, discriminador.trainable_variables)
    opt_g.apply_gradients(zip(grad_g, generador.trainable_variables))    # 5. cada red actualiza solo sus pesos
    opt_d.apply_gradients(zip(grad_d, discriminador.trainable_variables))
    return loss_g, loss_d

**Lo importante:** el gradiente del generador **atraviesa el discriminador** (porque su pérdida depende de lo que el discriminador opina), pero solo se actualizan los pesos del generador. Así el falsificador aprende qué cambiar para engañar mejor.

## Paso 4.6. Entrenar y mirar cómo mejora (lámina 127)

Usamos siempre **el mismo ruido** para mostrar el progreso: así vemos cómo evolucionan las mismas 8 "semillas". Con GPU, cada época tarda unos 10 a 20 segundos.

In [ ]:
EPOCAS_GAN = 15
ruido_fijo = tf.random.normal((8, RUIDO), seed=1)
historial = {"generador": [], "discriminador": []}
muestras = {}

inicio = time.time()
for epoca in range(1, EPOCAS_GAN + 1):
    for lote in datos_gan:
        lg, ld = paso(lote)
    historial["generador"].append(float(lg)); historial["discriminador"].append(float(ld))
    if epoca in (1, 3, 8, EPOCAS_GAN):
        muestras[epoca] = (generador(ruido_fijo, training=False).numpy() + 1) / 2
    print(f"Época {epoca:2d} | pérdida G {float(lg):.3f} | pérdida D {float(ld):.3f} | {time.time() - inicio:.0f} s")

mostrar(list(muestras.values()), [f"Época {e}" for e in muestras])

**Qué deberías ver:** en la época 1, manchas; en la 3, siluetas; desde la 8, prendas reconocibles: camisetas, pantalones, zapatos y bolsos **que no existen** en el dataset.

## Paso 4.7. Las pérdidas (lámina 128)

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(historial["generador"], label="Generador"); plt.plot(historial["discriminador"], label="Discriminador")
plt.xlabel("Época"); plt.ylabel("Pérdida"); plt.legend(); plt.grid(alpha=0.3); plt.title("Pérdidas de la GAN"); plt.show()

**Cómo leerlas:** en una GAN, **las pérdidas no bajan hasta cero** como en un clasificador, porque las dos redes compiten. Lo sano es que **se estabilicen** en valores parecidos. Si la del discriminador llega a casi 0, ganó la competencia y el generador deja de aprender. Por eso, en GAN, la evaluación principal es **mirar las imágenes**.

### Ejercicio 8
Genera **16 prendas nuevas** con ruido al azar y muéstralas con `mostrar`. Guarda las imágenes generadas, ya pasadas al rango de 0 a 1, en `nuevas` (un arreglo de NumPy de forma (16, 28, 28, 1)).

In [ ]:
nuevas = None  # TU CÓDIGO AQUÍ

In [ ]:
revisar(lambda: nuevas.shape == (16, 28, 28, 1) and nuevas.min() >= 0 and nuevas.max() <= 1)

### Ejercicio 9
**¿El discriminador se deja engañar?** Pasa 1.000 imágenes reales y 1.000 falsas por el discriminador, y calcula qué proporción de las **falsas** clasifica como reales (logit mayor que 0, que equivale a probabilidad mayor que 0,5). Guárdalo en `prop_engañado`.

In [ ]:
prop_engañado = None  # TU CÓDIGO AQUÍ
print(prop_engañado)

In [ ]:
revisar(lambda: prop_engañado is not None and 0 <= prop_engañado <= 1)

**Cómo leerlo:** si está cerca de 0, el discriminador todavía descubre casi todas las falsas: al generador le falta. Si está cerca de 0,5, el discriminador ya no las distingue: **la GAN llegó al equilibrio** que se busca en teoría (lámina 103).

## Paso 4.8. Moverse por el espacio de ruido

Como cada imagen sale de un vector de ruido, podemos ir **de un vector a otro de a poco** y ver cómo una prenda se transforma en otra. Es la prueba de que el generador aprendió algo continuo, no solo copió imágenes.

In [ ]:
z1, z2 = tf.random.normal((1, RUIDO), seed=3), tf.random.normal((1, RUIDO), seed=7)
pasos = tf.concat([(1 - a) * z1 + a * z2 for a in np.linspace(0, 1, 8)], axis=0)
mostrar([(generador(pasos, training=False).numpy() + 1) / 2], ["De un ruido a otro, en 8 pasos"])

### Ejercicio 10
¿Qué cambiarías si la GAN generara imágenes borrosas o siempre la misma prenda? Escribe dos ideas en la celda siguiente. (No tiene verificación.)

**Mis ideas:** (doble clic para escribir)

---
# Cierre

| Red | Qué aprende | Para qué sirve |
|---|---|---|
| Autoencoder | A comprimir y reconstruir | Representaciones compactas, quitar ruido, **detectar anomalías** |
| GAN | A generar datos nuevos que parecen reales | Imágenes sintéticas, aumentar datos, arte, diseño |

**Para el capstone:**

- **Si su problema es detectar casos raros** (fraudes, fallas, defectos) y tienen pocos ejemplos anómalos, un autoencoder entrenado con lo normal es una muy buena opción.
- **Las GAN** son difíciles de entrenar y de evaluar. Para generar imágenes o texto en un proyecto, hoy conviene partir de modelos ya entrenados (Capítulos 10 y 12).

Guarden el notebook en `ejercicios_estudiantes` como `clase17_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
784 pixeles en 32 números: 24,5 veces más pequeño.

In [ ]:
factor_compresion = 784 / CODIGO
print(factor_compresion)

### Ejercicio 2
El codificador es un modelo por sí solo: recibe imágenes de 784 pixeles y entrega códigos de 32 números.

In [ ]:
codigos_8 = codificador.predict(X_te[:8], verbose=0)
print(codigos_8.shape)
print(codigos_8[0].round(2))

### Ejercicio 3

In [ ]:
limpiadas = limpiador.predict(X_te_ruido[:1000], verbose=0)
mse_ruido = np.mean((X_te_ruido[:1000] - X_te[:1000]) ** 2)
mse_limpia = np.mean((limpiadas - X_te[:1000]) ** 2)
print(round(mse_ruido, 4), round(mse_limpia, 4), "| el error baja a", round(mse_limpia / mse_ruido, 2), "veces el original")

### Ejercicio 4
`groupby(y_te)` agrupa los errores por prenda; `drop(NORMAL)` quita las zapatillas; `idxmin` da el número de la prenda con menor error.

In [ ]:
por_prenda = pd.Series(errores).groupby(y_te).mean().drop(NORMAL)
prenda_mas_parecida = PRENDAS[por_prenda.idxmin()]
print(prenda_mas_parecida)
print(por_prenda.rename(index=dict(enumerate(PRENDAS))).sort_values().round(4))

### Ejercicio 5
Para detectar más anomalías, el umbral debe **bajar**: más imágenes superan el umbral y se marcan. El percentil 80 está más abajo que el 95, así que sube el recall y también las falsas alarmas (cerca de un 20% de las zapatillas).

In [ ]:
umbral_80 = np.percentile(errores_normales_tr, 80)
alarma_80 = (errores > umbral_80).astype(int)
recall_p80 = recall_score(es_anomalia, alarma_80)
print(round(recall_p80, 3), "| falsas alarmas:", round(alarma_80[es_anomalia == 0].mean(), 3))

### Ejercicio 6
Con y = 0, la fórmula queda −log(1 − p). Engañado: −log(0,1) = 2,30. Acierta: −log(0,9) = 0,11.

In [ ]:
bce_engañado = -np.log(1 - 0.9)
bce_acierta = -np.log(1 - 0.1)
print(round(bce_engañado, 3), round(bce_acierta, 3))

### Ejercicio 7
64 filtros × (4 × 4 × 1 pesos + 1 sesgo) = 64 × 17 = 1.088.

In [ ]:
params_conv_d = 64 * (4 * 4 * 1 + 1)
print(params_conv_d, discriminador.layers[0].count_params())

### Ejercicio 8
El generador entrega valores entre −1 y 1 (por la tanh); se suma 1 y se divide por 2 para pasarlos a 0 y 1.

In [ ]:
nuevas = (generador(tf.random.normal((16, RUIDO)), training=False).numpy() + 1) / 2
mostrar([nuevas[:8], nuevas[8:]])

### Ejercicio 9

In [ ]:
reales = X_gan[:1000]
falsas = generador(tf.random.normal((1000, RUIDO)), training=False)
prop_engañado = float(np.mean(discriminador(falsas, training=False).numpy() > 0))
prop_reales_ok = float(np.mean(discriminador(reales, training=False).numpy() > 0))
print("Falsas que pasan por reales:", round(prop_engañado, 3), "| Reales reconocidas:", round(prop_reales_ok, 3))

### Ejercicio 10
Ideas razonables: entrenar más épocas; si siempre sale la misma prenda (*colapso de modo*), bajar la tasa de aprendizaje del generador o agregar ruido a las etiquetas del discriminador; si salen borrosas, agregar una capa convolucional más al generador; o usar una GAN condicional, que recibe además la clase de prenda a generar.